# Microproyecto 2 - Analisis de sentimientos: procesamiento de lenguaje natural

### Ricardo Jose Garzon, Luisa Fernanda Aristizabal y Maria Fernanda Hurtado

**Deep Learning - 2026. Semestre 2**

### Introduccion 

aqui va la introduccion 

### 1. Importaciones

In [1]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [2]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split


### 2. cargado, analisis y preprocesamiento de datos

In [3]:
df = pd.read_csv("movie.csv")
df.head()


,text,label
0,I grew up (b. 1965) watching and loving the Th...,0
1,"When I put this movie in my DVD player, and sa...",0
2,Why do people who do not know what a particula...,0
3,Even though I have great interest in Biblical ...,0
4,Im a die hard Dads Army fan and nothing will e...,1


Con este info podemos observar 1. que no existen valores nulos, el dataset esta "limpio" estructuralmente, ya tenemos la variable objetivo lista

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    40000 non-null  str  
 1   label   40000 non-null  int64
dtypes: int64(1), str(1)
memory usage: 625.1 KB


In [5]:
df['label'].value_counts()

label
0    20019
1    19981
Name: count, dtype: int64

**Revisión de duplicados**

El dataset IMDB suele contener reseñas repetidas. Si quedan en train y test contaminan la evaluación, así que las eliminamos antes de cualquier análisis posterior.

In [6]:
print("Duplicados antes:", df.duplicated(subset="text").sum())
df = df.drop_duplicates(subset="text").reset_index(drop=True)
print("Filas tras eliminar duplicados:", len(df))
print(df["label"].value_counts())

Duplicados antes: 277
Filas tras eliminar duplicados: 39723
label
1    19908
0    19815
Name: count, dtype: int64


In [7]:
# Longitud de las reseñas
df['length'] = df['text'].apply(lambda x: len(x.split()))

# Revisar percentiles
print(df['length'].describe())
print(df['length'].quantile([0.50, 0.75, 0.90, 0.95]))

count    39723.000000
mean       231.486142
std        171.367657
min          4.000000
25%        126.000000
50%        173.000000
75%        282.000000
max       2470.000000
Name: length, dtype: float64
0.50    173.0
0.75    282.0
0.90    453.0
0.95    590.0
Name: length, dtype: float64


Con esto podemos observar diferentes comportamientos 1. hay mucha variabilidad, ya que algunas resenas pueden ser muy cortas (4 palabras) otras son demasiado largas (2470). La media es de 231, la mediana es de 173, como la media es mayor, podemos concluir que hay outliers largos (resenas enormes). Ahora 50% de reseñas ≤ 173 palabras 75% ≤ 282 palabras. 

**Se definió una longitud máxima de secuencia de 300 palabras para el modelo, valor que cubre aproximadamente el percentil 80 de la distribución de longitudes. Esto permite capturar la mayor parte del contenido relevante sin incrementar innecesariamente el costo computacional. Las reseñas más largas son truncadas, mientras que las más cortas son rellenadas mediante padding.**

El uso de un tamaño de secuencia fijo es necesario debido a las restricciones de entrada de las redes neuronales, particularmente en modelos recurrentes como LSTM, que requieren entradas de dimensión uniforme.


In [8]:
max_len =  300
max_words = 10000

**LIMPIEZA DE TEXTO**

In [9]:
def clean_text(text):
    text = BeautifulSoup(text, "html.parser").get_text()
    text = text.lower()
    text = re.sub(r"[^a-zA-Z']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)

df[["text", "clean_text", "label"]].head()

,text,clean_text,label
0,I grew up (b. 1965) watching and loving the Th...,i grew up b watching and loving the thunderbir...,0
1,"When I put this movie in my DVD player, and sa...",when i put this movie in my dvd player and sat...,0
2,Why do people who do not know what a particula...,why do people who do not know what a particula...,0
3,Even though I have great interest in Biblical ...,even though i have great interest in biblical ...,0
4,Im a die hard Dads Army fan and nothing will e...,im a die hard dads army fan and nothing will e...,1


**Verificación de reseñas vacías tras la limpieza**

El regex puede dejar strings vacíos si una reseña original contenía solo HTML o símbolos. Las eliminamos para evitar entradas vacías al tokenizador.

In [10]:
vacios = (df["clean_text"].str.strip() == "").sum()
print("Reseñas vacías tras limpieza:", vacios)

df = df[df["clean_text"].str.strip() != ""].reset_index(drop=True)
print("Filas finales:", len(df))

Reseñas vacías tras limpieza: 0
Filas finales: 39723


In [11]:
X_text = df["clean_text"]
y = df["label"].values

In [12]:
X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    X_text,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(X_train_text.shape)
print(X_val_text.shape)

(27806,)
(5958,)


**Tokenización y padding**

El tokenizador se ajusta **únicamente con el conjunto de entrenamiento** para evitar fugas de información (data leakage). Se reserva un token `<OOV>` para palabras no vistas que aparezcan en validación o test.

In [13]:
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_val_seq   = tokenizer.texts_to_sequences(X_val_text)
X_test_seq  = tokenizer.texts_to_sequences(X_test_text)

print("Ejemplo secuencia (primeros 20 tokens):", X_train_seq[0][:20])

Ejemplo secuencia (primeros 20 tokens): [68, 10, 37, 4891, 1, 59, 7, 4, 309, 759, 3, 32, 567, 4667, 3, 8805, 1730, 40, 112, 8]


**Padding a longitud fija (`max_len = 300`)**

Las reseñas más cortas se rellenan con ceros y las más largas se truncan, ambas operaciones al final de la secuencia (`post`).

In [14]:
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
X_val_pad   = pad_sequences(X_val_seq,   maxlen=max_len, padding="post", truncating="post")
X_test_pad  = pad_sequences(X_test_seq,  maxlen=max_len, padding="post", truncating="post")

print("Shape train:", X_train_pad.shape)
print("Shape val:  ", X_val_pad.shape)
print("Shape test: ", X_test_pad.shape)

Shape train: (27806, 300)
Shape val:   (5958, 300)
Shape test:  (5959, 300)


**Tamaño del vocabulario**

Valor que se usará en la capa `Embedding` del modelo RNN.

In [15]:
vocab_size = min(max_words, len(tokenizer.word_index) + 1)
print("vocab_size:", vocab_size)
print("Total palabras únicas vistas en train:", len(tokenizer.word_index))

vocab_size: 10000
Total palabras únicas vistas en train: 93559


 **Pipeline de datos**

Se construye un pipeline que entregará los datos al modelo RNN por lotes (batches), con `shuffle` solo en entrenamiento y `prefetch` para solapar I/O y cómputo. Validación y test no se mezclan para que las métricas sean reproducibles.

In [16]:
import tensorflow as tf

BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

train_pipeline = (
    tf.data.Dataset.from_tensor_slices((X_train_pad, y_train))
    .shuffle(buffer_size=len(X_train_pad), seed=42)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_pipeline = (
    tf.data.Dataset.from_tensor_slices((X_val_pad, y_val))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_pipeline = (
    tf.data.Dataset.from_tensor_slices((X_test_pad, y_test))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("train_pipeline:", train_pipeline)
print("val_pipeline:  ", val_pipeline)
print("test_pipeline: ", test_pipeline)

train_pipeline: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 300), dtype=tf.int32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>
val_pipeline:   <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 300), dtype=tf.int32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>
test_pipeline:  <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 300), dtype=tf.int32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>


**Verificación rápida del pipeline**

Tomamos un batch para confirmar shapes y dtypes antes de pasarlo al modelo.

In [17]:
for x_batch, y_batch in train_pipeline.take(1):
    print("X batch shape:", x_batch.shape, "dtype:", x_batch.dtype)
    print("y batch shape:", y_batch.shape, "dtype:", y_batch.dtype)

print("Batches por epoch (train):", len(train_pipeline))
print("Batches por epoch (val):  ", len(val_pipeline))
print("Batches por epoch (test): ", len(test_pipeline))

X batch shape: (64, 300) dtype: <dtype: 'int32'>
y batch shape: (64,) dtype: <dtype: 'int64'>
Batches por epoch (train): 435
Batches por epoch (val):   94
Batches por epoch (test):  94


### 3.0 Configuración de rendimiento

Antes de construir el modelo configuramos:

1. **Semillas** (`tf.keras.utils.set_random_seed`) para reproducibilidad entre corridas.
2. **Detección automática de GPU** vía `tf.config.list_physical_devices('GPU')`. Si hay GPU NVIDIA con CUDA, TensorFlow la usará por defecto en cualquier llamada a `model.fit` / `predict`. Si solo hay CPU, también funciona — solo más lento.
3. **Mixed precision (`mixed_float16`)** automática **si se detecta GPU**: las multiplicaciones de matrices usan `float16` (mucho más rápidas en GPUs NVIDIA con Tensor Cores) y la capa final se queda en `float32` para estabilidad numérica. En CPU se desactiva (no aporta) y el modelo entrena en `float32`.
4. **`BATCH_SIZE = 256`**: sweet spot para esta arquitectura. Triplica el throughput frente a batch 64.
5. **`.cache()` en los `tf.data.Dataset`**: tras el primer epoch, los tensores ya padded quedan en RAM (~47 MB para 39 723 reseñas × 300 ints), evitando recrearlos cada epoch.
6. **`recurrent_dropout=0`** en el LSTM (al construir el modelo): mantener este valor activa la ruta acelerada del kernel (cuDNN en GPU NVIDIA, kernel rápido en CPU).


In [18]:
# 1) Reproducibilidad
tf.keras.utils.set_random_seed(42)

# 2) Detección automática de hardware
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU detectada: {gpus[0].name}")
    # Habilitar memory growth para no acaparar toda la VRAM
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass   # ya inicializada
    # 3) Mixed precision: float16 en matmuls, float32 en la capa de salida
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print(f"Mixed precision policy: {tf.keras.mixed_precision.global_policy().name}")
else:
    print("Sin GPU — entrenando en CPU (float32)")
    tf.keras.mixed_precision.set_global_policy("float32")

# 4) Reconstruir los pipelines con cache() y batch_size mayor
BATCH_SIZE = 256
AUTOTUNE = tf.data.AUTOTUNE

train_pipeline = (
    tf.data.Dataset.from_tensor_slices((X_train_pad, y_train))
    .cache()
    .shuffle(buffer_size=len(X_train_pad), seed=42)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
val_pipeline = (
    tf.data.Dataset.from_tensor_slices((X_val_pad, y_val))
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
test_pipeline = (
    tf.data.Dataset.from_tensor_slices((X_test_pad, y_test))
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(f"\nBATCH_SIZE: {BATCH_SIZE}")
print(f"Batches por epoch (train): {len(train_pipeline)}")
print(f"Batches por epoch (val):   {len(val_pipeline)}")
print(f"Batches por epoch (test):  {len(test_pipeline)}")


CPUs: ['/physical_device:CPU:0']
GPUs: ninguna (CPU only)

BATCH_SIZE: 256
Batches por epoch (train): 109
Batches por epoch (val):   24
Batches por epoch (test):  24


### 3. Arquitectura del modelo: Bi-LSTM + Attention

La arquitectura elegida combina tres componentes que atacan dificultades concretas del análisis de sentimientos sobre reseñas largas:

1. **Embedding aprendible** — convierte cada token (entero) en un vector denso. Reemplaza el one-hot disperso de 10 000 dimensiones por uno de 128 dimensiones donde palabras semánticamente parecidas tienden a quedar cerca.
2. **LSTM Bidireccional** — lee la secuencia hacia adelante y hacia atrás. Cada palabra obtiene contexto de lo que vino *antes* y *después*, lo que captura mejor construcciones como `"not good"` o `"could have been better"`.
3. **Attention aditiva (Bahdanau)** — en lugar de quedarse solo con el último estado oculto, aprende a ponderar la importancia de **cada paso temporal** y produce un vector resumen. Esto evita el cuello de botella del LSTM en reseñas largas y hace al modelo interpretable (los pesos de atención son visualizables).

Vamos a construirla **capa por capa** para entender cada parte.


#### 3.1 Capa de `Embedding`

**Qué hace.** Cada token de entrada es un entero entre 0 y `vocab_size − 1` (en nuestro caso, 0..9999). La capa `Embedding` mantiene una matriz aprendible de tamaño `(vocab_size, embedding_dim)` y, dado un token `i`, devuelve la **fila `i`** de esa matriz como vector denso.

```
token id (entero)  →  vector denso de dimensión 128
       42          →  [-0.12, 0.31, ..., 0.04]
```

**Por qué no usar one-hot.** Un one-hot de 10 000 dimensiones es disperso, no captura similitud semántica y es caro de propagar. El embedding aprende **representaciones distribuidas** durante el entrenamiento: tras converger, palabras como `"great"` y `"excellent"` terminan con vectores cercanos porque aparecen en contextos similares dentro de las reseñas positivas.

**Parámetros que usaremos.**

| Parámetro | Valor | Razón |
|---|---|---|
| `input_dim` | `vocab_size` (10 000) | Tamaño del vocabulario que tokenizamos |
| `output_dim` | 128 | Compromiso entre capacidad y costo. Valores típicos: 50–300 |
| `mask_zero` | `True` | El padding usa el ID 0; con la máscara las capas siguientes lo ignoran y no contamina los gradientes |

**Costo en parámetros.** `10 000 × 128 = 1 280 000` parámetros entrenables — la mayoría del modelo va a estar en esta capa.

**Forma de los tensores.**

```
entrada:  (batch, max_len)        → (64, 300)         dtype int32
salida:   (batch, max_len, dim)   → (64, 300, 128)    dtype float32
```


In [19]:
# Imports adicionales para construir el modelo
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM, Dense, Dropout, Layer
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import numpy as np
print("Keras layers listas. vocab_size =", vocab_size, "| max_len =", max_len)


Keras layers listas. vocab_size = 10000 | max_len = 300


#### 3.2 Capa `Bidirectional(LSTM)`

**Qué resuelve.** Una LSTM unidireccional procesa la secuencia de izquierda a derecha y, en el paso `t`, su estado oculto `h_t` contiene información de los tokens `0..t` pero **no** de lo que viene después. Para sentimiento esto es problemático: en *"not particularly good"*, cuando el LSTM ve la palabra `"not"` no sabe si modifica algo positivo o negativo. La **versión bidireccional** corre dos LSTMs en paralelo:

- una hacia adelante (`→`) que produce `h_t^→` (contexto de `0..t`),
- otra hacia atrás (`←`) que produce `h_t^←` (contexto de `t..T`),

y concatena ambas: `h_t = [h_t^→ ; h_t^←]`. Cada palabra obtiene representación con contexto **completo** a ambos lados.

**`return_sequences=True`** — clave para Attention.

| Configuración | Salida |
|---|---|
| `return_sequences=False` (default) | Solo el último estado: `(batch, 2·units)` → `(256, 128)` |
| `return_sequences=True` | Un estado por paso: `(batch, max_len, 2·units)` → `(256, 300, 128)` |

Necesitamos `True` porque Attention, en la siguiente sección, debe ponderar **todos** los pasos temporales. Si solo le diéramos el último estado, no habría a qué prestarle atención.

**Por qué `units=64`.** Cada dirección produce 64 dimensiones. Tras la concatenación, la salida del Bi-LSTM tiene dimensión **128**, igual a la del Embedding — un tamaño cómodo y manejable.

**Por qué `recurrent_dropout=0`.** Keras tiene una **ruta acelerada** del LSTM (kernel "fused") que solo se activa si:
- `activation='tanh'` (default) ✓
- `recurrent_activation='sigmoid'` (default) ✓
- `recurrent_dropout=0` ← **importante**, no lo subas
- `use_bias=True`, `unroll=False` (defaults) ✓

Si pones `recurrent_dropout > 0`, Keras cae a una implementación genérica que es notoriamente más lenta. Para regularizar usamos solo `dropout=0.3` (dropout sobre la entrada del paso, no sobre la conexión recurrente). En la práctica esto basta y mantenemos la velocidad.

**Conteo de parámetros.** Una `LSTM(64)` con entrada de dimensión 128 tiene 4 compuertas, cada una con: matriz de entrada `W` de tamaño `(128, 64)`, matriz recurrente `U` de tamaño `(64, 64)`, y un sesgo de tamaño `(64,)`. Total por dirección:

`4 × (128·64 + 64·64 + 64) = 4 × 12 352 = 49 408` parámetros.

Bidireccional → `2 × 49 408 = 98 816` parámetros.

**Forma de los tensores en nuestro modelo.**

```
entrada:  (256, 300, 128)   ← salida del Embedding
salida:   (256, 300, 128)   ← (forward 64) ⊕ (backward 64)
```


#### 3.3 Capa de `Attention` (aditiva, estilo Bahdanau)

**Por qué implementarla a mano.** Keras ofrece `tf.keras.layers.Attention` y `AdditiveAttention`, pero ambas están pensadas para arquitecturas con **query/key/value** separados (como en seq2seq o Transformers). Aquí queremos algo más simple: dado un tensor `(batch, time, features)`, producir un único vector `(batch, features)` ponderando los pasos temporales según su importancia para la clasificación. Implementarla nosotras también es más didáctico — vas a ver exactamente qué parámetros aprende y por qué.

**Fórmula (Bahdanau aditiva).** Para cada paso `t = 0..299`:

$$
u_t = \tanh(W\,h_t + b) \quad
\text{score}_t = v^\top u_t \quad
\alpha_t = \frac{\exp(\text{score}_t)}{\sum_{k} \exp(\text{score}_k)} \quad
c = \sum_{t} \alpha_t\, h_t
$$

Donde:
- `h_t` ∈ ℝ¹²⁸ es la salida del Bi-LSTM en el paso `t`.
- `W` ∈ ℝ¹²⁸ˣ⁶⁴, `b` ∈ ℝ⁶⁴, `v` ∈ ℝ⁶⁴ son **parámetros aprendibles**.
- `score_t` es un escalar; `α_t` el peso de atención normalizado vía softmax.
- `c` ∈ ℝ¹²⁸ es el **vector de contexto**: el resumen de la reseña que pasará al clasificador.

**Intuición.** El modelo descubre solo qué busca: la matriz `W` proyecta cada estado oculto a un espacio donde resulta fácil "comparar" qué tan relevante es; el vector `v` es la dirección hacia la que apuntan los estados importantes; el `tanh` introduce no-linealidad para que pueda aprender relaciones no triviales (por ejemplo, ponderar más palabras emocionalmente cargadas).

**El detalle crítico — masking.** Las reseñas cortas se rellenaron con `0` hasta llegar a 300 tokens. Si dejamos que el softmax considere esas posiciones, repartirá probabilidad sobre el padding y diluirá los pesos reales. Para evitarlo, **antes** del softmax sumamos `-1e9` a los scores donde la máscara dice "padding" — eso los manda al `-∞` práctico y `softmax` les asigna `α ≈ 0`. La máscara la propaga el `Embedding(mask_zero=True)` automáticamente.

**Bonus — interpretabilidad.** El tensor `α` de forma `(batch, 300)` es directamente visualizable: puedes mapear el peso de cada token sobre la reseña original y ver qué palabras "miró" el modelo para predecir positivo/negativo. Es una herramienta gratuita de explicabilidad.

**Conteo de parámetros (con `units=64`).**

| Pesos | Shape | Parámetros |
|---|---|---|
| `W` | (128, 64) | 8 192 |
| `b` | (64,) | 64 |
| `v` | (64, 1) | 64 |
| **Total** | | **8 320** |

Mucho menos que el Bi-LSTM, pero su impacto es alto porque es la capa que aprende **qué mirar**.


In [20]:
class AttentionLayer(Layer):
    """Self-attention aditiva (Bahdanau) para colapsar (batch, time, feat) → (batch, feat).

    Aprende W, b, v y respeta la máscara de padding propagada desde el Embedding.
    Devuelve (context, alpha) — el primero es el resumen, el segundo los pesos
    de atención por paso temporal (útiles para visualización).
    """

    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.supports_masking = True   # propaga la máscara desde el Embedding

    def build(self, input_shape):
        feat_dim = input_shape[-1]
        self.W = self.add_weight(
            name="W", shape=(feat_dim, self.units),
            initializer="glorot_uniform", trainable=True,
        )
        self.b = self.add_weight(
            name="b", shape=(self.units,),
            initializer="zeros", trainable=True,
        )
        self.v = self.add_weight(
            name="v", shape=(self.units, 1),
            initializer="glorot_uniform", trainable=True,
        )
        super().build(input_shape)

    def call(self, inputs, mask=None):
        # inputs: (batch, time, feat)
        # 1) u_t = tanh(W·h_t + b)
        u = tf.tanh(tf.matmul(inputs, self.W) + self.b)            # (batch, time, units)
        # 2) score_t = v^T · u_t
        scores = tf.squeeze(tf.matmul(u, self.v), axis=-1)         # (batch, time)
        # 3) masked softmax: -1e9 en posiciones de padding
        if mask is not None:
            mask_f = tf.cast(mask, scores.dtype)
            scores = scores + (1.0 - mask_f) * -1e9
        alpha = tf.nn.softmax(scores, axis=-1)                     # (batch, time)
        # 4) context = sum_t alpha_t · h_t
        context = tf.reduce_sum(inputs * tf.expand_dims(alpha, -1), axis=1)  # (batch, feat)
        return context, alpha

    def compute_mask(self, inputs, mask=None):
        # tras colapsar el eje temporal ya no propagamos máscara
        return None

    def get_config(self):
        cfg = super().get_config()
        cfg["units"] = self.units
        return cfg


### 4. Construcción del modelo: GloVe + Bi-LSTM + Attention + Ensemble

Aplicamos cuatro mejoras combinadas para alcanzar el techo realista de la arquitectura RNN sobre IMDB:

| # | Componente | Razón |
|---|---|---|
| 1 | **Embeddings GloVe 6B 200d** pre-entrenados | 6 mil millones de tokens — ~300 000× más texto que el corpus IMDB |
| 2 | **`max_len=500`** y **`vocab=20 000`** | Cubrir el percentil 92 de longitudes, conservar más palabras informativas |
| 3 | **Búsqueda de hiperparámetros** (4 variantes vs baseline) | Validar si el baseline puede mejorarse con cambios de `lstm_units`, dropout o lr |
| 4 | **Ensemble de 5 modelos** con seeds aleatorias distintas | Reducir varianza promediando predicciones — gana +0.5 a +1.0 puntos típicamente |

**Flujo de la sección.**

1. **4.1–4.3**: re-tokenizar, cargar GloVe, definir `build_model`.
2. **4.4 Baseline**: entrenar **una vez** con config sensata y mínimo 15 epochs → primera referencia.
3. **4.5 HP search**: 4 variantes × 5 epochs (rápido) — comparadas contra el baseline.
4. **4.6 Ensemble**: 5 seeds del config ganador (baseline o variante) → resultado final.

> **Sobre el techo.** El estado del arte RNN puro en IMDB ronda **0.92**. Para superar 0.93 hay que cambiar a Transformer/BERT, lo que ya no es RNN.


#### 4.1 Re-tokenización con vocab más amplio y secuencias más largas

`max_len=500` cubre el percentil ~92 de la distribución de longitudes (vs. ~80 de los 300 anteriores), perdiendo menos contexto en reseñas largas. `vocab_size=20 000` mantiene el doble de palabras informativas — nombres de actores, géneros, jerga — que antes se mapeaban al token `<OOV>`.

El tokenizador se reentrena solo en `X_train_text` (sin leakage). Las pipelines se reconstruyen con `BATCH_SIZE=256` y `.cache()`.


In [21]:
# Hiperparámetros nuevos
max_len = 500
max_words = 20000

# Re-tokenizar (solo train; sin leakage)
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_val_seq   = tokenizer.texts_to_sequences(X_val_text)
X_test_seq  = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
X_val_pad   = pad_sequences(X_val_seq,   maxlen=max_len, padding="post", truncating="post")
X_test_pad  = pad_sequences(X_test_seq,  maxlen=max_len, padding="post", truncating="post")

vocab_size = min(max_words, len(tokenizer.word_index) + 1)
print(f"vocab_size: {vocab_size:,}  |  max_len: {max_len}")
print(f"Shapes — train: {X_train_pad.shape}  val: {X_val_pad.shape}  test: {X_test_pad.shape}")

# Reconstruir pipelines
BATCH_SIZE = 256
AUTOTUNE = tf.data.AUTOTUNE

train_pipeline = (
    tf.data.Dataset.from_tensor_slices((X_train_pad, y_train))
    .cache().shuffle(len(X_train_pad), seed=42).batch(BATCH_SIZE).prefetch(AUTOTUNE)
)
val_pipeline = (
    tf.data.Dataset.from_tensor_slices((X_val_pad, y_val))
    .cache().batch(BATCH_SIZE).prefetch(AUTOTUNE)
)
test_pipeline = (
    tf.data.Dataset.from_tensor_slices((X_test_pad, y_test))
    .cache().batch(BATCH_SIZE).prefetch(AUTOTUNE)
)
print(f"Batches por epoch (train): {len(train_pipeline)}")


vocab_size: 20,000  |  max_len: 500
Shapes — train: (27806, 500)  val: (5958, 500)  test: (5959, 500)
Batches por epoch (train): 109


#### 4.2 Cargar GloVe pre-entrenado (6B tokens, 200 dimensiones)

GloVe ([Pennington et al. 2014](https://nlp.stanford.edu/projects/glove/)) entrena embeddings sobre estadísticas globales de co-ocurrencia. Usamos la versión `glove-wiki-gigaword-200` — 200 dimensiones, vocabulario de 400k palabras, entrenado sobre **6 mil millones de tokens** (Wikipedia + Gigaword). Eso es ~300 000× más texto que nuestras 27k reseñas, y de ahí viene la ganancia.

Lo descargamos vía `gensim.downloader` (~250 MB, una vez; luego queda cacheado en `~/gensim-data/`).


In [22]:
import gensim.downloader as api

# Descargar/cachear GloVe (la primera vez tarda ~2-3 min, luego es instantáneo)
print("Cargando GloVe glove-wiki-gigaword-200...")
glove = api.load("glove-wiki-gigaword-200")
print(f"Vectores GloVe disponibles: {len(glove):,}")
print(f"Dimensión: {glove.vector_size}")

# Verificación cualitativa
for word in ["good", "bad", "boring", "brilliant", "terrible"]:
    if word in glove:
        sims = glove.most_similar(word, topn=5)
        print(f"  {word!r:>10s} → {[(w, round(s,3)) for w, s in sims]}")


Cargando GloVe glove-wiki-gigaword-200...
Vectores GloVe disponibles: 400,000
Dimensión: 200
      'good' → [('better', 0.814), ('really', 0.802), ('always', 0.791), ('sure', 0.779), ('you', 0.775)]
       'bad' → [('good', 0.711), ('things', 0.708), ('worse', 0.7), ('really', 0.69), ('because', 0.686)]
    'boring' → [('tedious', 0.706), ('bored', 0.651), ('dull', 0.641), ('monotonous', 0.609), ('downright', 0.584)]
  'brilliant' → [('superb', 0.725), ('dazzling', 0.685), ('brilliantly', 0.64), ('wonderful', 0.597), ('bright', 0.586)]
  'terrible' → [('horrible', 0.89), ('awful', 0.832), ('dreadful', 0.739), ('horrendous', 0.722), ('horrific', 0.704)]


**Construir la matriz de embeddings alineada con el tokenizador.** Mismo procedimiento que con W2V: cada palabra del tokenizador busca su vector en GloVe; las que no están quedan en ceros.


In [23]:
EMBED_DIM = 200
embedding_matrix = np.zeros((vocab_size, EMBED_DIM), dtype=np.float32)
covered = 0
not_in_glove = []

for word, idx in tokenizer.word_index.items():
    if idx >= vocab_size:
        continue
    if word in glove:
        embedding_matrix[idx] = glove[word]
        covered += 1
    else:
        not_in_glove.append(word)

total = vocab_size - 1
print(f"Cobertura GloVe: {covered:,}/{total:,} = {covered/total*100:.1f}%")
print(f"Palabras no presentes: {len(not_in_glove):,}")
print(f"Ejemplos: {not_in_glove[:10]}")
print(f"Shape de la matriz: {embedding_matrix.shape}")


Cobertura GloVe: 19,212/19,999 = 96.1%
Palabras no presentes: 787
Ejemplos: ['<OOV>', "it's", "don't", "i'm", "doesn't", "didn't", "can't", "that's", "i've", "isn't"]
Shape de la matriz: (20000, 200)


#### 4.3 `build_model`: con opción de Bi-LSTM apilado

Generalizamos `build_model` para soportar **2 capas de Bi-LSTM apiladas** opcionalmente. Cuando `stacked=True`, la primera Bi-LSTM devuelve secuencias y la segunda también, dándole a la atención una representación más profunda. Cuesta ~50% más en tiempo de entrenamiento.


In [24]:
def build_model(
    vocab_size, max_len,
    embedding_matrix=None,
    embedding_dim=200,
    lstm_units=64,
    attn_units=64,
    dropout1=0.3,
    dropout2=0.2,
    stacked=False,
    embedding_trainable=True,
):
    inputs = Input(shape=(max_len,), dtype="int32", name="tokens")

    if embedding_matrix is not None:
        emb_layer = Embedding(
            input_dim=vocab_size, output_dim=embedding_dim,
            embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
            mask_zero=True, trainable=embedding_trainable, name="embedding",
        )
    else:
        emb_layer = Embedding(vocab_size, embedding_dim, mask_zero=True, name="embedding")

    x = emb_layer(inputs)
    x = Bidirectional(
        LSTM(lstm_units, return_sequences=True, dropout=0.3, recurrent_dropout=0.0),
        name="bilstm_1",
    )(x)
    if stacked:
        x = Bidirectional(
            LSTM(lstm_units, return_sequences=True, dropout=0.3, recurrent_dropout=0.0),
            name="bilstm_2",
        )(x)

    context, alpha = AttentionLayer(units=attn_units, name="attention")(x)
    h = Dropout(dropout1, name="dropout_1")(context)
    h = Dense(64, activation="relu", name="dense_1")(h)
    h = Dropout(dropout2, name="dropout_2")(h)
    output = Dense(1, activation="sigmoid", dtype="float32", name="output")(h)  # float32 explícito para estabilidad bajo mixed precision

    model = Model(inputs, output, name="bilstm_attention")
    attn_model = Model(inputs, [output, alpha], name="bilstm_attention_with_alpha")
    return model, attn_model

print("build_model actualizada con opción stacked=True/False")


build_model actualizada con opción stacked=True/False


#### 4.4 Entrenamiento inicial (baseline)

Antes de la búsqueda, entrenamos **una vez** el modelo con hiperparámetros razonables y **15+ epochs garantizadas** (`start_from_epoch=15`). Esto nos da:

1. **Una referencia rápida** del nivel que ya alcanzamos solo con GloVe + arquitectura base.
2. **Un punto de comparación** para saber si la búsqueda de HPs aporta mejora real.

**Config baseline**: `lstm_units=64`, `dropout=0.3/0.2`, `lr=1e-3`, sin apilar.


In [25]:
tf.keras.utils.set_random_seed(42)

baseline_hp = {"lstm_units": 64, "dropout1": 0.3, "dropout2": 0.2, "lr": 1e-3, "stacked": False, "name": "baseline"}

model, attention_model = build_model(
    vocab_size=vocab_size, max_len=max_len,
    embedding_matrix=embedding_matrix, embedding_dim=EMBED_DIM,
    lstm_units=baseline_hp["lstm_units"],
    dropout1=baseline_hp["dropout1"],
    dropout2=baseline_hp["dropout2"],
    stacked=baseline_hp["stacked"],
)
model.compile(
    optimizer=Adam(learning_rate=baseline_hp["lr"]),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
)

es_baseline = EarlyStopping(
    monitor="val_loss", patience=4, restore_best_weights=True,
    start_from_epoch=15, verbose=1,
)
rlr_baseline = ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1,
)

print(f"=== Entrenando BASELINE ({baseline_hp['name']}) — mínimo 15 epochs ===")
history = model.fit(
    train_pipeline, validation_data=val_pipeline,
    epochs=25, callbacks=[es_baseline, rlr_baseline], verbose=1,
)

baseline_val_acc = max(history.history["val_accuracy"])
baseline_val_auc = max(history.history["val_auc"])
baseline_val_loss = min(history.history["val_loss"])
print(f"\n>>> BASELINE: val_acc={baseline_val_acc:.4f}  val_auc={baseline_val_auc:.4f}  val_loss={baseline_val_loss:.4f}")


=== Entrenando BASELINE (baseline) — mínimo 15 epochs ===
Epoch 1/25
109/109 ━━━━━━━━━━━━━━━━━━━━ 141s 1s/step - accuracy: 0.7484 - auc: 0.8416 - loss: 0.4867 - val_accuracy: 0.8733 - val_auc: 0.9494 - val_loss: 0.3101 - learning_rate: 0.0010
Epoch 2/25
109/109 ━━━━━━━━━━━━━━━━━━━━ 142s 1s/step - accuracy: 0.8824 - auc: 0.9505 - loss: 0.2845 - val_accuracy: 0.9015 - val_auc: 0.9632 - val_loss: 0.2490 - learning_rate: 0.0010
Epoch 3/25
109/109 ━━━━━━━━━━━━━━━━━━━━ 144s 1s/step - accuracy: 0.9149 - auc: 0.9698 - loss: 0.2199 - val_accuracy: 0.9072 - val_auc: 0.9668 - val_loss: 0.2419 - learning_rate: 0.0010
Epoch 4/25
109/109 ━━━━━━━━━━━━━━━━━━━━ 140s 1s/step - accuracy: 0.9345 - auc: 0.9798 - loss: 0.1775 - val_accuracy: 0.9067 - val_auc: 0.9665 - val_loss: 0.2487 - learning_rate: 0.0010
Epoch 5/25
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9476 - auc: 0.9861 - loss: 0.1445
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
109/109 ━━━━━━━━━━━━━━━━━━━

#### 4.5 Búsqueda de hiperparámetros (grid reducido)

Para validar si existe una mejor config que el baseline, probamos **4 variantes** sobre las decisiones más impactantes. Cada una entrena solo **5 epochs** con `EarlyStopping(patience=2)` — suficiente para identificar tendencias relativas sin gastar mucho tiempo (la convergencia completa se hace en el ensemble).

| Variable | Valores |
|---|---|
| `lstm_units` | 64, 128 |
| `dropout1 / dropout2` | (0.3, 0.2), (0.4, 0.3) |
| `learning_rate` | 1e-3, 5e-4 |
| `stacked` | False (apilar Bi-LSTMs cuesta +50% tiempo) |

El ranking se compara contra el `baseline_val_acc` para decidir si vale la pena cambiar de config.


In [26]:
hp_grid = [
    {"name": "lstm 128",     "lstm_units": 128, "dropout1": 0.3, "dropout2": 0.2, "lr": 1e-3, "stacked": False},
    {"name": "dropout +",    "lstm_units": 64,  "dropout1": 0.4, "dropout2": 0.3, "lr": 1e-3, "stacked": False},
    {"name": "lr 5e-4",      "lstm_units": 64,  "dropout1": 0.3, "dropout2": 0.2, "lr": 5e-4, "stacked": False},
    {"name": "lstm128 lr 5e-4 dropout+", "lstm_units": 128, "dropout1": 0.4, "dropout2": 0.3, "lr": 5e-4, "stacked": False},
]

results = [{"name": "baseline", **{k: baseline_hp[k] for k in ['lstm_units','dropout1','dropout2','lr','stacked']},
            "val_accuracy": baseline_val_acc, "val_loss": baseline_val_loss, "epochs_run": len(history.history['val_loss'])}]

for i, hp in enumerate(hp_grid):
    print(f"\n=== {i+1}/{len(hp_grid)}  {hp['name']}  {hp} ===")
    tf.keras.utils.set_random_seed(42)
    m, _ = build_model(
        vocab_size=vocab_size, max_len=max_len,
        embedding_matrix=embedding_matrix, embedding_dim=EMBED_DIM,
        lstm_units=hp["lstm_units"], dropout1=hp["dropout1"], dropout2=hp["dropout2"],
        stacked=hp["stacked"],
    )
    m.compile(
        optimizer=Adam(learning_rate=hp["lr"]),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    cb = [EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True, verbose=0)]
    h = m.fit(train_pipeline, validation_data=val_pipeline, epochs=5, callbacks=cb, verbose=0)
    best_va = max(h.history["val_accuracy"])
    best_vl = min(h.history["val_loss"])
    epochs_run = len(h.history["val_loss"])
    print(f"   epochs corridos: {epochs_run}  |  best val_acc: {best_va:.4f}  |  best val_loss: {best_vl:.4f}")
    results.append({**hp, "val_accuracy": best_va, "val_loss": best_vl, "epochs_run": epochs_run})

df_results = pd.DataFrame(results).sort_values("val_accuracy", ascending=False).reset_index(drop=True)
print("\n=== RANKING (incluye baseline) ===")
print(df_results[["name", "lstm_units", "dropout1", "dropout2", "lr", "stacked", "val_accuracy", "val_loss", "epochs_run"]].to_string(index=False))

best_hp = df_results.iloc[0].to_dict()
if best_hp["name"] == "baseline":
    print(f"\n>>> Ningún cambio mejora al baseline. Usaremos baseline en el ensemble.")
else:
    print(f"\n>>> Mejor config: '{best_hp['name']}' (val_acc={best_hp['val_accuracy']:.4f}, baseline={baseline_val_acc:.4f}, Δ={best_hp['val_accuracy']-baseline_val_acc:+.4f})")



=== 1/4  lstm 128  {'name': 'lstm 128', 'lstm_units': 128, 'dropout1': 0.3, 'dropout2': 0.2, 'lr': 0.001, 'stacked': False} ===


KeyboardInterrupt: 